In [1]:
# Test that run pod connection is working
import torch                                                                                                                               
print(f"PyTorch: {torch.__version__}")                                                                                                   
print(f"CUDA available: {torch.cuda.is_available()}")                                                                                      
print(f"GPU: {torch.cuda.get_device_name(0)}")                                                                                           
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
VRAM: 25.3 GB


In [ ]:
from huggingface_hub import login
login()

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig                                                                           
import torch                                                                                                                                             
                                                                                                                                                            
model_id = "google/gemma-2-9b"                                                                                                                           

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded.")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


Loading weights: 100%|██████████| 464/464 [00:08<00:00, 56.57it/s] 


Model loaded.
VRAM used: 8.4 GB


In [6]:
import json                                                                                                                                                
                                                                                                                                                    
# Load one story from the dataset
with open("stories.jsonl") as f:
    story = json.loads(f.readline())

print(f"Story: {story['id']}")
print(f"Domain: {story['domain']} | Condition: {story['condition']}")
print(f"\n{story['story'][:300]}...")

# Tokenize
inputs = tokenizer(story["story"], return_tensors="pt").to("cuda")
print(f"\nTokens: {inputs['input_ids'].shape[1]}")

Story: hiring_01_biased
Domain: hiring | Condition: biased

Marcus had been through enough hiring cycles to trust his first read. He picked up the next folder: Darius Williams. Computer science from State, four years at a cloud infrastructure company out of Atlanta, GitHub handle printed neatly at the top.

He opened the GitHub tab. Genuine commit history, c...

Tokens: 346


In [7]:
with torch.no_grad():                                                                                                                      
    outputs = model(                                                                                                                     
        **inputs,
        output_hidden_states=True,
    )

hidden_states = outputs.hidden_states
print(f"Number of layers: {len(hidden_states)}")
print(f"Shape of each layer's activations: {hidden_states[0].shape}")
print(f"  (batch_size, sequence_length, hidden_dim)")

Number of layers: 43
Shape of each layer's activations: torch.Size([1, 346, 3584])
  (batch_size, sequence_length, hidden_dim)


In [8]:
for i, hs in enumerate(hidden_states[::10]):  # every 10th layer
    print(f"Layer {i*10:2d} | mean norm: {hs.norm(dim=-1).mean().item():.2f}")


Layer  0 | mean norm: 97.51
Layer 10 | mean norm: 123.83
Layer 20 | mean norm: 240.71
Layer 30 | mean norm: 429.19
Layer 40 | mean norm: 741.50
